# PPO 16 x 16

In [2]:
# =========================================================
# PPO COMPARATIVE ANALYSIS READY CODE
# FINAL STABLE VERSION
# =========================================================

import numpy as np
import tkinter as tk
from tkinter import messagebox
import requests
import time
import psutil

# =========================================================
# PARAMETERS
# =========================================================

GRID_SIZE = 16

ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]

gamma = 0.99
actor_lr = 0.005
critic_lr = 0.01
clip_epsilon = 0.2

# =========================================================
# FAIR COMPARISON
# =========================================================

episodes = 3000

# =========================================================

TILE_SIZE_CM = 50

start = None
goal = None

obstacles = []

mode = "obstacle"

buttons = {}
tile_entry = None

# =========================================================
# COMPARATIVE ANALYSIS VARIABLES
# =========================================================

episode_rewards = []

steps_per_episode = []

success_count = 0

# =========================================================
# PPO POLICY TABLE
# =========================================================

policy = np.ones(
    (GRID_SIZE, GRID_SIZE, len(ACTIONS))
) / 4

# =========================================================
# STATE VALUE TABLE
# =========================================================

V = np.zeros((GRID_SIZE, GRID_SIZE))

# =========================================================

def manhattan(a, b):

    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# =========================================================

def softmax(x):

    exp_x = np.exp(x - np.max(x))

    return exp_x / np.sum(exp_x)

# =========================================================

def step(state, action):

    r, c = state

    moves = {
        0: (-1, 0),
        1: (1, 0),
        2: (0, -1),
        3: (0, 1)
    }

    dr, dc = moves[action]

    nr = r + dr
    nc = c + dc

    # WALL COLLISION

    if nr < 0 or nr >= GRID_SIZE or nc < 0 or nc >= GRID_SIZE:

        return state, -10, False

    new_state = (nr, nc)

    # OBSTACLE HIT

    if new_state in obstacles:

        return new_state, -100, True

    # GOAL REACHED

    if new_state == goal:

        return new_state, 500, True

    reward = -1

    old_dist = manhattan(state, goal)

    new_dist = manhattan(new_state, goal)

    # REWARD SHAPING

    if new_dist < old_dist:
        reward += 5
    else:
        reward -= 5

    return new_state, reward, False

# =========================================================

def train():

    global policy, V, TILE_SIZE_CM
    global success_count

    if start is None or goal is None:

        messagebox.showerror(
            "Error",
            "Set Start and Goal first"
        )

        return

    TILE_SIZE_CM = float(tile_entry.get())

    # =====================================================
    # TRAINING TIMER START
    # =====================================================

    start_time = time.time()

    # =====================================================

    for ep in range(episodes):

        trajectory = []

        state = start

        visited = set()

        total_reward = 0

        for t in range(300):

            r, c = state

            probs = softmax(policy[r, c])

            action = np.random.choice(
                4,
                p=probs
            )

            old_prob = probs[action]

            new_state, reward, done = step(
                state,
                action
            )

            # LOOP PENALTY

            if new_state in visited:
                reward -= 20
            else:
                visited.add(new_state)

            nr, nc = new_state

            # TD TARGET

            td_target = reward + gamma * V[nr, nc]

            # ADVANTAGE

            advantage = td_target - V[r, c]

            trajectory.append((
                state,
                action,
                reward,
                old_prob,
                advantage,
                new_state
            ))

            # CRITIC UPDATE

            V[r, c] += critic_lr * advantage

            state = new_state

            total_reward += reward

            if done:

                if state == goal:
                    success_count += 1

                break

        # =================================================
        # PPO POLICY UPDATE
        # =================================================

        for data in trajectory:

            (
                state,
                action,
                reward,
                old_prob,
                advantage,
                next_state
            ) = data

            r, c = state

            probs = softmax(policy[r, c])

            new_prob = probs[action]

            ratio = new_prob / (
                old_prob + 1e-8
            )

            clipped_ratio = np.clip(
                ratio,
                1 - clip_epsilon,
                1 + clip_epsilon
            )

            ppo_objective = min(
                ratio * advantage,
                clipped_ratio * advantage
            )

            policy[r, c, action] += (
                actor_lr * ppo_objective
            )

        # =================================================
        # COMPARATIVE ANALYSIS TRACKING
        # =================================================

        episode_rewards.append(total_reward)

        steps_per_episode.append(t + 1)

    # =====================================================
    # TRAINING TIMER END
    # =====================================================

    end_time = time.time()

    training_time = (
        end_time - start_time
    )

    # =====================================================
    # RESOURCE USAGE
    # =====================================================

    cpu_usage = psutil.cpu_percent()

    ram_usage = psutil.virtual_memory().percent

    # =====================================================
    # PERFORMANCE METRICS
    # =====================================================

    avg_reward = np.mean(
        episode_rewards
    )

    avg_steps = np.mean(
        steps_per_episode
    )

    success_rate = (
        success_count / episodes
    ) * 100

    # =====================================================
    # FINAL RESULTS
    # =====================================================

    print("\n==============================")
    print("PPO COMPARATIVE RESULTS")
    print("==============================")

    print(
        f"Training Time: "
        f"{training_time:.2f} sec"
    )

    print(
        f"CPU Usage: "
        f"{cpu_usage}%"
    )

    print(
        f"RAM Usage: "
        f"{ram_usage}%"
    )

    print(
        f"Average Reward: "
        f"{avg_reward:.2f}"
    )

    print(
        f"Average Steps: "
        f"{avg_steps:.2f}"
    )

    print(
        f"Success Rate: "
        f"{success_rate:.2f}%"
    )

    print("==============================")

    # =====================================================

    print("\nVALUE TABLE\n")

    print(V)

    final_policy = extract_policy()

    simulate_path(final_policy)

    send_policy(final_policy)

# =========================================================

def extract_policy():

    final_policy = {}

    print("\nPOLICY\n")

    for r in range(GRID_SIZE):

        for c in range(GRID_SIZE):

            state = (r, c)

            if state in obstacles:
                continue

            if state == goal:

                final_policy[str(state)] = "GOAL"

                print(state, "-> GOAL")

                continue

            action = np.argmax(
                policy[r, c]
            )

            final_policy[str(state)] = (
                ACTIONS[action]
            )

            print(
                state,
                "->",
                ACTIONS[action]
            )

    return final_policy

# =========================================================

def simulate_path(policy_map):

    state = start

    path = [state]

    visited = set()

    steps = 0

    for _ in range(500):

        if state == goal:
            break

        if state in visited:

            print("Loop detected")

            break

        visited.add(state)

        if str(state) not in policy_map:

            print(
                "No policy for state:",
                state
            )

            break

        action = policy_map[str(state)]

        r, c = state

        if action == "UP":
            r -= 1

        elif action == "DOWN":
            r += 1

        elif action == "LEFT":
            c -= 1

        elif action == "RIGHT":
            c += 1

        next_state = (r, c)

        # OBSTACLE SAFETY

        if next_state in obstacles:

            print(
                "Obstacle encountered"
            )

            break

        state = next_state

        path.append(state)

        steps += 1

    distance = steps * TILE_SIZE_CM

    print("\nPATH:")

    print(path)

    print("\nSteps:", steps)

    print(
        "Tile Size:",
        TILE_SIZE_CM,
        "cm"
    )

    print(
        "Total Distance:",
        distance,
        "cm"
    )

# =========================================================

def send_policy(policy_map):

    state = start

    actions = []

    visited = set()

    for _ in range(500):

        if state == goal:

            actions.append("GOAL")

            break

        if state in visited:
            break

        visited.add(state)

        if str(state) not in policy_map:
            break

        action = policy_map[str(state)]

        actions.append(action)

        r, c = state

        if action == "UP":
            r -= 1

        elif action == "DOWN":
            r += 1

        elif action == "LEFT":
            c -= 1

        elif action == "RIGHT":
            c += 1

        next_state = (r, c)

        if next_state in obstacles:
            break

        state = next_state

    data = {
        "tile_size": TILE_SIZE_CM,
        "path": actions
    }

    url = "http://192.168.4.1/policy"

    try:

        requests.post(
            url,
            json=data
        )

        print("\nPath sent to ESP32")

        print(actions)

    except:

        print("\nESP32 not connected")

# =========================================================

def cell_click(r, c):

    global start, goal

    if mode == "start":

        if start:

            buttons[start].config(
                bg="white",
                text=""
            )

        start = (r, c)

        buttons[(r, c)].config(
            bg="blue",
            fg="white",
            text="S"
        )

    elif mode == "goal":

        if goal:

            buttons[goal].config(
                bg="white",
                text=""
            )

        goal = (r, c)

        buttons[(r, c)].config(
            bg="green",
            fg="white",
            text="G"
        )

    elif mode == "obstacle":

        if (r, c) not in obstacles:

            obstacles.append((r, c))

        buttons[(r, c)].config(
            bg="black",
            fg="white",
            text="X"
        )

    elif mode == "erase":

        if (r, c) in obstacles:

            obstacles.remove((r, c))

        if start == (r, c):
            start = None

        if goal == (r, c):
            goal = None

        buttons[(r, c)].config(
            bg="white",
            text=""
        )

# =========================================================

def set_mode(m):

    global mode

    mode = m

# =========================================================

def build_gui():

    global tile_entry

    root = tk.Tk()

    root.title(
        "16x16 PPO Robot Trainer"
    )

    control = tk.Frame(root)

    control.pack()

    tk.Button(
        control,
        text="Set Start",
        command=lambda:
        set_mode("start")
    ).grid(row=0, column=0)

    tk.Button(
        control,
        text="Set Goal",
        command=lambda:
        set_mode("goal")
    ).grid(row=0, column=1)

    tk.Button(
        control,
        text="Add Obstacle",
        command=lambda:
        set_mode("obstacle")
    ).grid(row=0, column=2)

    tk.Button(
        control,
        text="Erase",
        command=lambda:
        set_mode("erase")
    ).grid(row=0, column=3)

    tk.Button(
        control,
        text="Train PPO",
        command=train
    ).grid(row=0, column=4)

    tk.Label(
        control,
        text="Tile Size (cm)"
    ).grid(row=1, column=0)

    tile_entry = tk.Entry(
        control,
        width=10
    )

    tile_entry.insert(0, "40")

    tile_entry.grid(row=1, column=1)

    grid_frame = tk.Frame(root)

    grid_frame.pack()

    for r in range(GRID_SIZE):

        for c in range(GRID_SIZE):

            b = tk.Button(
                grid_frame,
                width=3,
                height=1,
                bg="white",
                command=lambda r=r, c=c:
                cell_click(r, c)
            )

            b.grid(row=r, column=c)

            buttons[(r, c)] = b

    root.mainloop()

# =========================================================

build_gui()


PPO COMPARATIVE RESULTS
Training Time: 5.04 sec
CPU Usage: 17.1%
RAM Usage: 46.2%
Average Reward: 499.39
Average Steps: 26.06
Success Rate: 96.60%

VALUE TABLE

[[440.44247408   0.          -3.99811672  -2.19764143  -4.07080198
   -5.37764993  -4.95323423  -6.50215737  -7.69086507  -7.05302678
   -4.61144524  -6.10436786  -6.11506749  -6.12718251  -4.83674989
   -6.56164249]
 [454.08881326   0.          -2.58597213  -3.14195342  -4.80147235
   -3.8164382   -4.3444461   -6.10690176  -6.9013127   -7.03914441
   -6.06206583  -6.39491502  -7.26494984  -5.36029765  -4.20724092
   -5.05873378]
 [464.49617669   0.          -5.6475849   -4.1220165   -5.71351923
   -4.97005989  -5.73374414  -5.76803782  -7.65750991  -8.21677678
   -7.23421198  -5.78727991  -5.63219359  -5.75897593  -4.47914926
   -5.84687449]
 [472.14152847   0.          -4.00247643  -4.32293277  -5.00053331
   -8.52779469  -8.66294872  -6.43491555  -8.62122443  -8.39746771
   -7.58644777  -6.70995127  -6.33478243  -5.57342982